In [8]:
from pathlib import Path
import sys

project_root=Path.cwd().parents[1]
sys.path.append(str(project_root))

In [9]:
from sentence_transformers import CrossEncoder

from src.retrieval.models import (RetrievedDocument, SearchCandidate)


In [10]:
class CrossEncoderReranker:
    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        self.model = CrossEncoder(model_name)

    def rerank(
        self,
        query: str,
        documents: list[SearchCandidate],
    ) -> list[SearchCandidate]:

        if not documents:
            return []

        pairs = [
            (query, doc.text)
            for doc in documents
        ]

        scores = self.model.predict(pairs)

        for doc, score in zip(documents, scores):
            doc.reranker_score = float(score)

        return documents